# LeetCode 1423 — Maximum Points You Can Obtain from Cards bằng Python

Link đề: https://leetcode.com/problems/maximum-points-you-can-obtain-from-cards/ · Độ khó: **Medium** · Chủ đề: Array, Sliding Window, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def maxScore(self, cardPoints: list[int], k: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra rằng **thứ tự** lấy bài không quan trọng — chỉ quan trọng **lấy bao nhiêu lá bên trái** (`i`), phần còn lại `k − i` lá bên phải.
2. Từ đó giảm bài toán `2^k` khả năng xuống còn `k + 1` khả năng, rồi tính mỗi khả năng `O(1)` bằng **prefix sum** hoặc **cập nhật trượt**.
3. Đổi góc nhìn: lấy `k` lá ở hai đầu ⇔ **bỏ lại một đoạn liên tiếp dài `n − k`** ở giữa ⇒ bài **cửa sổ trượt cố định** tìm tổng nhỏ nhất.
4. Biết vì sao tham lam "đầu nào lớn lấy đầu đó" là **sai**, và có phản ví dụ chạy thật.
5. Tránh các bẫy Python: `arr[-0:]` là **cả mảng**, vòng lặp thiếu trường hợp `i = k`, `k = n` (cửa sổ rỗng).

## 2. Đề bài & ví dụ

Có một hàng lá bài, lá thứ `i` có số điểm `cardPoints[i]`. Mỗi bước bạn lấy **một lá ở đầu trái hoặc đầu phải** của hàng. Bạn phải lấy **đúng `k` lá**. Điểm = tổng điểm các lá đã lấy. Trả về điểm **lớn nhất** có thể.

**Ví dụ 1:** `cardPoints = [1,2,3,4,5,6,1], k = 3` → `12` (lấy 3 lá bên phải: `1 + 6 + 5`).

**Ví dụ 2:** `cardPoints = [2,2,2], k = 2` → `4`.

**Ví dụ 3:** `cardPoints = [9,7,7,9,7,7,9], k = 7` → `55` (phải lấy hết).

**Ràng buộc:** `1 <= cardPoints.length <= 10^5`, `1 <= cardPoints[i] <= 10^4`, `1 <= k <= cardPoints.length`.

In [1]:
import random
import time
from itertools import accumulate
from functools import lru_cache
from typing import List

examples = [(([1, 2, 3, 4, 5, 6, 1], 3), 12), (([2, 2, 2], 2), 4), (([9, 7, 7, 9, 7, 7, 9], 7), 55)]
for (cards, k), want in examples:
    print(cards, "k =", k, "->", want)

[1, 2, 3, 4, 5, 6, 1] k = 3 -> 12
[2, 2, 2] k = 2 -> 4
[9, 7, 7, 9, 7, 7, 9] k = 7 -> 55


## 3. Kiến thức nền

### 3.1 Cắt lát (slicing) và chỉ số âm

`a[:i]` là `i` phần tử đầu; `a[-j:]` là `j` phần tử cuối — **nhưng** `a[-0:]` bằng `a[0:]` = **cả mảng** (vì `-0 == 0`). Đây là bẫy kinh điển; an toàn hơn là `a[len(a) - j:]`.

In [2]:
a = [1, 2, 3, 4, 5]
print(a[:2], a[-2:], a[len(a) - 2:])
print("a[-0:] =", a[-0:], "  a[len(a)-0:] =", a[len(a) - 0:])

[1, 2] [4, 5] [4, 5]
a[-0:] = [1, 2, 3, 4, 5]   a[len(a)-0:] = []


### 3.2 Prefix sum (tổng tiền tố)

`pre[i] = a[0] + ... + a[i-1]`, `pre[0] = 0`. Tổng đoạn `a[l:r]` = `pre[r] − pre[l]` trong `O(1)`. `itertools.accumulate(a, initial=0)` dựng `pre` nhanh.

In [3]:
a = [1, 2, 3, 4, 5, 6, 1]
pre = list(accumulate(a, initial=0))
print(pre)
print("tổng a[2:5] =", pre[5] - pre[2], "=", sum(a[2:5]))
print("tổng 3 phần tử cuối =", pre[7] - pre[4])

[0, 1, 3, 6, 10, 15, 21, 22]
tổng a[2:5] = 12 = 12
tổng 3 phần tử cuối = 12


### 3.3 Cửa sổ trượt kích thước cố định

Tổng của mọi đoạn dài `w`: tính tổng đoạn đầu, rồi mỗi lần trượt sang phải 1 bước **cộng phần tử mới vào, trừ phần tử rơi ra**. Tổng cộng `O(n)` thay vì `O(n·w)`.

In [4]:
a, w = [1, 2, 3, 4, 5, 6, 1], 4
s = sum(a[:w]); sums = [s]
for r in range(w, len(a)):
    s += a[r] - a[r - w]
    sums.append(s)
print(sums, "min =", min(sums))

[10, 14, 18, 16] min = 10


### 3.4 Đệ quy có nhớ `lru_cache`

Để kiểm chứng "thứ tự không quan trọng", ta viết đệ quy thử **mọi** chuỗi lựa chọn trái/phải, trạng thái `(l, r)` = đoạn bài còn lại. `@lru_cache` nhớ kết quả mỗi trạng thái.

In [5]:
@lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)
print(fib(50), fib.cache_info().hits)

12586269025 48


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1, `k = 3`. Dù lấy theo thứ tự nào (T = trái, P = phải), tập lá cuối cùng chỉ phụ thuộc **số lần chọn T**:
- `TTT`: lấy `1,2,3` (tổng 6).
- `TTP`, `TPT`, `PTT`: đều lấy `1,2` bên trái và `1` bên phải (tổng 4).
- `TPP`, `PTP`, `PPT`: lấy `1` trái, `1,6` phải (tổng 8).
- `PPP`: lấy `1,6,5` (tổng 12).

⇒ `2^3 = 8` chuỗi lựa chọn nhưng chỉ có `k + 1 = 4` **kết quả khác nhau**. Đáp án = max(6, 4, 8, 12) = 12.

Nhìn ngược lại: lấy `i` lá trái và `3 − i` lá phải ⇔ **bỏ lại** đoạn liên tiếp `cardPoints[i : i + 4]` (dài `n − k = 4`). Tổng lấy = `tổng tất cả (22) − tổng đoạn bỏ lại`. Muốn lấy nhiều nhất ⇒ bỏ lại đoạn có tổng **nhỏ nhất**: các đoạn dài 4 có tổng `10, 14, 18, 16` ⇒ min 10 ⇒ `22 − 10 = 12` ✔.

In [6]:
cards, k = [1, 2, 3, 4, 5, 6, 1], 3
n = len(cards)
for i in range(k + 1):
    left, right = cards[:i], cards[n - (k - i):]
    kept = cards[i:i + n - k]
    print(f"i={i}: trái {left} + phải {right} = {sum(left) + sum(right):2d} | bỏ lại {kept} (tổng {sum(kept)})")

i=0: trái [] + phải [5, 6, 1] = 12 | bỏ lại [1, 2, 3, 4] (tổng 10)
i=1: trái [1] + phải [6, 1] =  8 | bỏ lại [2, 3, 4, 5] (tổng 14)
i=2: trái [1, 2] + phải [1] =  4 | bỏ lại [3, 4, 5, 6] (tổng 18)
i=3: trái [1, 2, 3] + phải [] =  6 | bỏ lại [4, 5, 6, 1] (tổng 16)


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** thử mọi `i` từ `0` đến `k` (số lá lấy bên trái), tính tổng bằng `sum` trên lát cắt. Đúng nhờ quan sát ở mục 4, nhưng mỗi lần tính lại tổng từ đầu.

In [7]:
def max_score_brute(cards, k):
    n = len(cards)
    best = 0
    for i in range(k + 1):
        s = sum(cards[:i]) + sum(cards[n - (k - i):])   # dùng n-(k-i), tránh bẫy -0
        best = max(best, s)
    return best

for (cards, k), want in examples:
    print(max_score_brute(cards, k), "✔" if max_score_brute(cards, k) == want else "✘")

12 ✔
4 ✔
55 ✔


**Độ phức tạp:** `k + 1` lựa chọn × `O(k)` cho mỗi lần `sum` ⇒ `O(k²)`. Với `k = 10^5` là `10^10` phép cộng ⇒ TLE.

**Brute force "thật sự"** (thử mọi chuỗi trái/phải) còn tệ hơn: `O(2^k)`. Ta viết nó (với `k` nhỏ) để **kiểm chứng** quan sát "thứ tự không quan trọng".

**Nút thắt:** tính lại tổng các lát cắt gần giống nhau ⇒ dùng **prefix sum** hoặc **cập nhật trượt**.

In [8]:
def max_score_all_orders(cards, k):
    """Thử mọi chuỗi lựa chọn trái/phải bằng đệ quy (không nhớ) — O(2^k)."""
    def go(l, r, left):
        if left == 0:
            return 0
        return max(cards[l] + go(l + 1, r, left - 1), cards[r] + go(l, r - 1, left - 1))
    return go(0, len(cards) - 1, k)

print([max_score_all_orders(c, k) for (c, k), _ in examples])
cards = list(range(1, 3001)); k = 3000
t = time.perf_counter(); max_score_brute(cards, k); print(f"brute O(k^2) với k=3000: {time.perf_counter() - t:.3f}s")

[12, 4, 55]
brute O(k^2) với k=3000: 0.066s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — chỉ có `k + 1` phương án:** mọi chuỗi lựa chọn đều cho tập lá dạng "`i` lá đầu + `k − i` lá cuối". (Chứng minh: lá lấy bên trái luôn là tiền tố liên tiếp, bên phải là hậu tố liên tiếp.)

**Insight 2 — tính mỗi phương án `O(1)`:**
- Prefix sum: `score(i) = pre[i] + (pre[n] − pre[n − (k − i)])`.
- Hoặc trượt: bắt đầu `i = k` (lấy `k` lá trái), mỗi bước **trả lại** lá trái thứ `i` và **lấy thêm** một lá phải. Nếu `i` giảm 1 thì `score` đổi `− cards[i−1] + cards[n − (k − i + 1)]`.

**Insight 3 — bài toán bù:** `score(i) = total − sum(cards[i : i + n − k])`. Max `score` ⇔ **min tổng cửa sổ dài `n − k`**. Quy tắc: **nếu** `k == n` **thì** đáp án là `total` (cửa sổ rỗng).

**Vì sao không tham lam?** "Lấy đầu lớn hơn" chỉ nhìn 1 bước; một lá nhỏ có thể che một lá rất lớn phía sau (ví dụ `[5, 1, 1, 100, 1]`, `k = 2`).

## 7. Các cách tiếp theo

### Cách 2 — Prefix sum

**Từng đoạn:**
1. `pre = [0] + tổng dồn` (độ dài `n + 1`).
2. Với `i` trong `0..k`: tổng trái `pre[i]`, tổng phải `pre[n] − pre[n − k + i]`.
3. Lấy max.

In [9]:
def max_score_prefix(cards, k):
    n = len(cards)
    pre = list(accumulate(cards, initial=0))
    return max(pre[i] + pre[n] - pre[n - k + i] for i in range(k + 1))

for (cards, k), want in examples:
    print(max_score_prefix(cards, k), max_score_prefix(cards, k) == want)

12 True
4 True
55 True


In [10]:
class SolutionPrefix:
    def maxScore(self, cardPoints: list[int], k: int) -> int:
        n = len(cardPoints)
        pre = [0] * (n + 1)
        for i, x in enumerate(cardPoints):
            pre[i + 1] = pre[i] + x
        return max(pre[i] + pre[n] - pre[n - k + i] for i in range(k + 1))

print([SolutionPrefix().maxScore(c, k) for (c, k), _ in examples])

[12, 4, 55]


**Độ phức tạp:** `O(n)` dựng prefix + `O(k)` duyệt ⇒ `O(n)` thời gian, `O(n)` bộ nhớ.

### Cách 3 — Trượt từ "tất cả bên trái" sang "tất cả bên phải" (`O(k)`, `O(1)` bộ nhớ)

**Từng đoạn:**
1. `cur = sum(cards[:k])` — phương án `i = k`.
2. Lặp `j = 1..k`: trả lại lá trái `cards[k − j]`, lấy thêm lá phải `cards[n − j]`: `cur += cards[n − j] − cards[k − j]`.
3. Mỗi bước cập nhật `best`.

In [11]:
def max_score_swap(cards, k):
    n = len(cards)
    cur = sum(cards[:k])
    best = cur
    for j in range(1, k + 1):
        cur += cards[n - j] - cards[k - j]
        best = max(best, cur)
    return best

for (cards, k), want in examples:
    print(max_score_swap(cards, k), max_score_swap(cards, k) == want)

12 True
4 True
55 True


In [12]:
class SolutionSwap:
    def maxScore(self, cardPoints: list[int], k: int) -> int:
        n = len(cardPoints)
        cur = best = sum(cardPoints[:k])
        for j in range(1, k + 1):
            cur += cardPoints[n - j] - cardPoints[k - j]
            best = max(best, cur)
        return best

print([SolutionSwap().maxScore(c, k) for (c, k), _ in examples])

[12, 4, 55]


**Độ phức tạp:** `O(k)` thời gian, `O(1)` bộ nhớ phụ — tốt nhất khi `k` nhỏ hơn nhiều so với `n`.

### Cách 4 — Cửa sổ trượt tìm đoạn **bỏ lại** có tổng nhỏ nhất

**Từng đoạn:**
1. `w = n − k`, `total = sum(cards)`. Nếu `w == 0` trả `total`.
2. `s = sum(cards[:w])`, `mn = s`.
3. Trượt `r` từ `w` đến `n − 1`: `s += cards[r] − cards[r − w]`, `mn = min(mn, s)`.
4. Trả `total − mn`.

In [13]:
def max_score_window(cards, k):
    n = len(cards)
    w = n - k
    total = sum(cards)
    if w == 0:
        return total
    s = mn = sum(cards[:w])
    for r in range(w, n):
        s += cards[r] - cards[r - w]
        mn = min(mn, s)
    return total - mn

for (cards, k), want in examples:
    print(max_score_window(cards, k), max_score_window(cards, k) == want)

12 True
4 True
55 True


In [14]:
class SolutionWindow:
    def maxScore(self, cardPoints: list[int], k: int) -> int:
        n = len(cardPoints)
        w = n - k
        total = sum(cardPoints)
        if w == 0:
            return total
        s = mn = sum(cardPoints[:w])
        for r in range(w, n):
            s += cardPoints[r] - cardPoints[r - w]
            mn = min(mn, s)
        return total - mn

print([SolutionWindow().maxScore(c, k) for (c, k), _ in examples])

[12, 4, 55]


**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ. Cách này cho thấy bài thực chất là **"Minimum Size Subarray Sum" kích thước cố định** — một pattern rất hay gặp.

### Cách 5 (kiểm chứng) — DP / đệ quy có nhớ trên `(l, r)`

Thử mọi chuỗi lựa chọn nhưng nhớ theo trạng thái `(l, số đã lấy)` (vì `r` suy ra được): `O(k²)` trạng thái. Không dùng để nộp, chỉ để thấy DP "đúng nhưng thừa" so với quan sát ở Insight 1.

In [15]:
def max_score_memo(cards, k):
    n = len(cards)
    @lru_cache(maxsize=None)
    def go(l, taken):
        if taken == k:
            return 0
        r = n - 1 - (taken - l)            # đã lấy l lá trái, taken-l lá phải
        return max(cards[l] + go(l + 1, taken + 1), cards[r] + go(l, taken + 1))
    return go(0, 0)

print([max_score_memo(c, k) for (c, k), _ in examples])

[12, 4, 55]


## 8. TRACE từng bước

Trace cách 3 (trượt) với `cards = [1,2,3,4,5,6,1], k = 3`, `n = 7`:

| bước j | trả lại `cards[k−j]` | lấy thêm `cards[n−j]` | `cur` | lá đang giữ | `best` |
|---|---|---|---|---|---|
| 0 | — | — | 6 | `1,2,3` | 6 |
| 1 | 3 | 1 | 4 | `1,2 | 1` | 6 |
| 2 | 2 | 6 | 8 | `1 | 6,1` | 8 |
| 3 | 1 | 5 | 12 | `| 5,6,1` | 12 |

In [16]:
def trace_swap(cards, k):
    n = len(cards)
    cur = best = sum(cards[:k])
    print(f"j=0: giữ trái {cards[:k]} phải [] -> cur={cur}, best={best}")
    for j in range(1, k + 1):
        out_card, in_card = cards[k - j], cards[n - j]
        cur += in_card - out_card
        best = max(best, cur)
        print(f"j={j}: trả lại {out_card}, lấy {in_card} -> giữ trái {cards[:k - j]} phải {cards[n - j:]} "
              f"cur={cur}, best={best}")
    return best

trace_swap([1, 2, 3, 4, 5, 6, 1], 3)

j=0: giữ trái [1, 2, 3] phải [] -> cur=6, best=6
j=1: trả lại 3, lấy 1 -> giữ trái [1, 2] phải [1] cur=4, best=6
j=2: trả lại 2, lấy 6 -> giữ trái [1] phải [6, 1] cur=8, best=8
j=3: trả lại 1, lấy 5 -> giữ trái [] phải [5, 6, 1] cur=12, best=12


12

Trace cách 4 (cửa sổ bỏ lại) cùng ví dụ, `w = 4`, `total = 22`:

In [17]:
def trace_window(cards, k):
    n = len(cards); w = n - k; total = sum(cards)
    print(f"total={total}, độ dài cửa sổ bỏ lại w={w}")
    if w == 0:
        print("w=0 -> lấy hết"); return total
    s = mn = sum(cards[:w])
    print(f"cửa sổ [0..{w - 1}] = {cards[:w]} tổng {s}, mn={mn}")
    for r in range(w, n):
        s += cards[r] - cards[r - w]
        mn = min(mn, s)
        print(f"trượt: +{cards[r]} -{cards[r - w]} -> cửa sổ [{r - w + 1}..{r}] = {cards[r - w + 1:r + 1]} "
              f"tổng {s}, mn={mn}")
    print("đáp án =", total, "-", mn, "=", total - mn)
    return total - mn

trace_window([1, 2, 3, 4, 5, 6, 1], 3)

total=22, độ dài cửa sổ bỏ lại w=4
cửa sổ [0..3] = [1, 2, 3, 4] tổng 10, mn=10
trượt: +5 -1 -> cửa sổ [1..4] = [2, 3, 4, 5] tổng 14, mn=10
trượt: +6 -2 -> cửa sổ [2..5] = [3, 4, 5, 6] tổng 18, mn=10
trượt: +1 -3 -> cửa sổ [3..6] = [4, 5, 6, 1] tổng 16, mn=10
đáp án = 22 - 10 = 12


12

## 9. Code nộp LeetCode

Dùng cách 4 (cửa sổ trượt, `O(n)` thời gian, `O(1)` bộ nhớ):

In [18]:
class Solution:
    def maxScore(self, cardPoints: list[int], k: int) -> int:
        n = len(cardPoints)
        w = n - k                          # độ dài đoạn bỏ lại ở giữa
        total = sum(cardPoints)
        if w == 0:
            return total
        s = mn = sum(cardPoints[:w])
        for r in range(w, n):
            s += cardPoints[r] - cardPoints[r - w]
            mn = min(mn, s)
        return total - mn

for (cards, k), want in examples:
    assert Solution().maxScore(cards, k) == want
print("3 ví dụ: OK")

3 ví dụ: OK


## 10. Vì sao thuật toán đúng

1. **Mọi cách lấy ⇒ dạng tiền tố + hậu tố:** các lá lấy từ bên trái luôn là `cards[0], cards[1], …` theo đúng thứ tự (lấy lá trái thì lá kế bên trở thành đầu trái mới), tức một tiền tố dài `i`; tương tự bên phải là hậu tố dài `k − i`. Hai phần không chồng nhau vì `i + (k − i) = k ≤ n`.
2. **Mọi `i ∈ [0, k]` đều thực hiện được:** lấy `i` lần bên trái rồi `k − i` lần bên phải.
3. Vậy đáp án = `max_i score(i)`. Cách 2 và 3 tính đúng từng `score(i)` (prefix sum / cập nhật trượt giữ bất biến `cur = score(k − j)`).
4. **Cách 4:** phần không bị lấy là đoạn liên tiếp `cards[i : i + n − k]`, nên `score(i) = total − sum(đoạn)`. Khi `i` chạy `0..k`, đoạn chạy qua **mọi** cửa sổ dài `n − k` ⇒ max `score` = `total − min cửa sổ`. Bất biến vòng lặp: `s` = tổng cửa sổ kết thúc tại `r`, `mn` = min các cửa sổ đã xét.

## 11. Phân tích Time / Space

Gọi `n = len(cardPoints)`.

| Cách | Time | Space |
|---|---|---|
| Thử mọi chuỗi trái/phải | `O(2^k)` | `O(k)` (đệ quy) |
| DP nhớ `(l, taken)` | `O(k²)` | `O(k²)` |
| 1. Brute `k+1` phương án + `sum` | `O(k²)` | `O(k)` (lát cắt) |
| 2. Prefix sum | `O(n)` | `O(n)` |
| 3. Trượt trái→phải | `O(k)` | `O(1)` |
| 4. Cửa sổ bỏ lại | `O(n)` | `O(1)` |

Cách 3 nhanh hơn cách 4 khi `k ≪ n`; cách 4 nhanh hơn khi `k` gần `n` (cửa sổ ngắn nhưng vẫn phải quét `n`). Cả hai đều đạt chuẩn.

In [19]:
rng = random.Random(1423)
big = [rng.randint(1, 10**4) for _ in range(10**5)]
for k in [10, 50000, 99990]:
    row = []
    for name, f in [("prefix", max_score_prefix), ("swap", max_score_swap), ("window", max_score_window)]:
        t = time.perf_counter(); r = f(big, k); row.append(f"{name}={time.perf_counter() - t:.3f}s")
    print(f"k={k:6d}: đáp án {r}, " + ", ".join(row))

k=    10: đáp án 57092, prefix=0.005s, swap=0.000s, window=0.002s


k= 50000: đáp án 251058099, prefix=0.008s, swap=0.005s, window=0.005s
k= 99990: đáp án 500340058, prefix=0.011s, swap=0.010s, window=0.008s


## 12. Bộ test

1. Ví dụ đề.
2. Edge: `n = 1`; `k = 1`; `k = n`; mọi phần tử bằng nhau; lá lớn ở giữa (không lấy được); lá lớn bị che sau lá nhỏ.
3. Đối chiếu tất cả cách với **thử mọi chuỗi lựa chọn** (`2^k`, `k ≤ 10`).
4. Ngẫu nhiên lớn hơn (đúng ràng buộc) giữa các cách `O(n)`.

In [20]:
impls = {
    "brute": max_score_brute, "prefix": max_score_prefix, "swap": max_score_swap,
    "window": max_score_window, "memo": max_score_memo,
    "Solution": lambda c, k: Solution().maxScore(c, k),
    "SolPrefix": lambda c, k: SolutionPrefix().maxScore(c, k),
    "SolSwap": lambda c, k: SolutionSwap().maxScore(c, k),
}
edge = [(([7], 1), 7), (([3, 9], 1), 9), (([1, 1000, 1], 2), 1001), (([1, 1, 1000, 1, 1], 2), 2),
        (([5, 1, 1, 100, 1], 2), 101), (([4, 4, 4, 4], 4), 16), (([100, 40, 17, 9, 73, 75], 3), 248)]
for name, f in impls.items():
    for (cards, k), want in examples + edge:
        assert f(cards, k) == want, (name, cards, k, f(cards, k))
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [21]:
rng = random.Random(23)
for _ in range(400):
    n = rng.randint(1, 10)
    cards = [rng.randint(1, 20) for _ in range(n)]
    k = rng.randint(1, n)
    want = max_score_all_orders(cards, k)
    for name, f in impls.items():
        assert f(cards, k) == want, (name, cards, k)
print("đối chiếu với thử mọi chuỗi trái/phải (2^k): OK")

for _ in range(200):
    n = rng.randint(1, 2000)
    cards = [rng.randint(1, 10**4) for _ in range(n)]
    k = rng.randint(1, n)
    a = max_score_prefix(cards, k)
    assert a == max_score_swap(cards, k) == max_score_window(cards, k) == Solution().maxScore(cards, k)
print("random n ≤ 2000: OK")

đối chiếu với thử mọi chuỗi trái/phải (2^k): OK


random n ≤ 2000: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam: mỗi bước lấy đầu lớn hơn

In [22]:
def wrong_greedy(cards, k):
    l, r, s = 0, len(cards) - 1, 0
    for _ in range(k):
        if cards[l] >= cards[r]:
            s += cards[l]; l += 1
        else:
            s += cards[r]; r -= 1
    return s

for cards, k in [([5, 1, 1, 100, 1], 2), ([1, 79, 80, 1, 1, 1, 200, 1], 3)]:
    print(cards, k, "| tham lam:", wrong_greedy(cards, k), "| đúng:", Solution().maxScore(cards, k))

[5, 1, 1, 100, 1] 2 | tham lam: 6 | đúng: 101
[1, 79, 80, 1, 1, 1, 200, 1] 3 | tham lam: 160 | đúng: 202


**Vì sao sai:** tham lam chỉ nhìn hai lá ở đầu. Trong `[5, 1, 1, 100, 1]`, lá `100` nằm sau lá `1` bên phải — tham lam lấy `5` trước rồi chỉ còn `1`, được `6`, trong khi lấy hai lá phải được `101`.

**Sửa:** xét **mọi** cách chia `i` trái / `k − i` phải (cách 2, 3, 4).

### Lỗi 2 — Bẫy `cards[-0:]` khi lấy 0 lá bên phải

In [23]:
def wrong_neg_zero(cards, k):
    best = 0
    for i in range(k + 1):
        best = max(best, sum(cards[:i]) + sum(cards[-(k - i):]))   # SAI khi k - i == 0
    return best

cards, k = [1, 2, 3, 4, 5, 6, 1], 3
print("sai:", wrong_neg_zero(cards, k), "| đúng:", Solution().maxScore(cards, k))
print("vì cards[-0:] =", cards[-0:])

sai: 28 | đúng: 12
vì cards[-0:] = [1, 2, 3, 4, 5, 6, 1]


**Vì sao sai:** khi `i = k`, `-(k - i) = 0` và `cards[-0:]` là **toàn bộ mảng** ⇒ cộng thừa, ra `6 + 22 = 28`.

**Sửa:** dùng `cards[len(cards) - (k - i):]` hoặc prefix sum `pre[n] - pre[n - (k - i)]`.

In [24]:
def fixed_neg_zero(cards, k):
    n = len(cards)
    return max(sum(cards[:i]) + sum(cards[n - (k - i):]) for i in range(k + 1))
print(fixed_neg_zero([1, 2, 3, 4, 5, 6, 1], 3))

12


### Lỗi 3 — Vòng lặp `range(k)` bỏ sót trường hợp lấy hết bên trái

In [25]:
def wrong_range(cards, k):
    n = len(cards)
    pre = list(accumulate(cards, initial=0))
    return max(pre[i] + pre[n] - pre[n - k + i] for i in range(k))   # SAI: thiếu i = k

cards, k = [9, 9, 1, 1, 1], 2
print("sai:", wrong_range(cards, k), "| đúng:", Solution().maxScore(cards, k))

sai: 10 | đúng: 18


**Vì sao sai:** số lá bên trái `i` nhận `k + 1` giá trị `0, 1, …, k`. `range(k)` dừng ở `k − 1`, bỏ mất phương án "lấy cả `k` lá bên trái" — đúng phương án tối ưu với `[9, 9, 1, 1, 1]`.

**Sửa:** `range(k + 1)`.

### Lỗi 4 — Tìm cửa sổ **lớn nhất** hoặc cửa sổ dài `k` thay vì `n − k`

In [26]:
def wrong_window_k(cards, k):
    # SAI: lấy max tổng đoạn liên tiếp dài k (nhưng đoạn lấy phải ở HAI ĐẦU)
    s = best = sum(cards[:k])
    for r in range(k, len(cards)):
        s += cards[r] - cards[r - k]
        best = max(best, s)
    return best

cards, k = [1, 1000, 1], 2
print("sai:", wrong_window_k([1, 1, 1000, 1, 1], 2), "| đúng:", Solution().maxScore([1, 1, 1000, 1, 1], 2))

sai: 1001 | đúng: 2


**Vì sao sai:** chỉ được lấy ở **hai đầu**, không lấy được đoạn giữa. Đoạn liên tiếp dài `k` ở giữa (chứa `1000`) không phải là phương án hợp lệ.

**Sửa:** đoạn **bỏ lại** mới là đoạn liên tiếp, dài `n − k`, và ta tìm tổng **nhỏ nhất** của nó.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Lấy từ **hai đầu** đúng `k` lần" ⇒ thứ tự không quan trọng, chỉ là **tiền tố + hậu tố** ⇒ `k + 1` phương án.
- "Tiền tố + hậu tố" ⇔ "**bỏ lại đoạn giữa**" ⇒ đổi max thành min trên cửa sổ dài `n − k` (bài 1658 Minimum Operations to Reduce X to Zero dùng đúng mẹo này).
- Khi DP trên `(l, r)` có vẻ tự nhiên nhưng `O(k²)` — hãy hỏi "kết quả cuối có phụ thuộc thứ tự không?".
- Trong Python, tránh `a[-j:]` khi `j` có thể bằng 0.
- Test nhanh: `k = n`, `n = 1`, lá lớn ở giữa, lá lớn sau lá nhỏ.

## 15. Biến thể của bài

1. **Lấy tối đa `k` lá** (có thể ít hơn), điểm có thể **âm** ⇒ phải xét mọi tổng số lá `t ≤ k`.
2. **LeetCode 1658** — Minimum Operations to Reduce X to Zero: bỏ hai đầu sao cho tổng bằng `x` ⇒ tìm đoạn giữa dài nhất có tổng `total − x`.
3. **LeetCode 1770** — Maximum Score from Performing Multiplication Operations: mỗi lần lấy nhân với hệ số khác nhau ⇒ thứ tự **quan trọng** ⇒ cần DP `(i, left)`.
4. **Hàng bài vòng tròn**: lấy `k` lá liên tiếp trên vòng tròn ⇒ cửa sổ trượt trên mảng nhân đôi.
5. **Hai người chơi lần lượt** lấy ở hai đầu (LeetCode 877, 486 Predict the Winner) ⇒ DP minimax.

Code biến thể 4 (vòng tròn, lấy `k` lá liên tiếp bất kỳ):

In [27]:
def max_circular_k(cards, k):
    n = len(cards)
    ext = cards + cards
    s = best = sum(ext[:k])
    for r in range(k, n + k - 1):
        s += ext[r] - ext[r - k]
        best = max(best, s)
    return best

print(max_circular_k([1, 2, 3, 4, 5, 6, 1], 3), "(lấy 2 đầu là một trường hợp riêng của vòng tròn)")

15 (lấy 2 đầu là một trường hợp riêng của vòng tròn)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute `k+1` phương án | `sum` lát cắt trái + phải | `O(k²)` | `O(k)` | Kiểm chứng, `k` nhỏ |
| 2. Prefix sum | `pre[i] + pre[n] − pre[n−k+i]` | `O(n)` | `O(n)` | Dễ nghĩ, dễ đúng |
| 3. Trượt trái→phải | Bắt đầu lấy hết trái, đổi dần sang phải | `O(k)` | `O(1)` | `k ≪ n`, gọn nhất |
| 4. Cửa sổ bỏ lại | `total − min` tổng đoạn dài `n − k` | `O(n)` | `O(1)` | Pattern tổng quát (1658) |
| 5. DP nhớ `(l, taken)` | Thử mọi chuỗi có nhớ | `O(k²)` | `O(k²)` | Khi thứ tự quan trọng (1770) |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().maxScore([1, 79, 80, 1, 1, 1, 200, 1], 3)` bằng bao nhiêu?

### Bài 2 (tính tay)
Lập bảng `score(i)` cho `i = 0..3` với `cards = [100, 40, 17, 9, 73, 75]`, `k = 3`. Đáp án?

### Bài 3 (điền TODO)
Hoàn thiện `best_split(cards, k)` trả về cặp `(điểm lớn nhất, số lá lấy bên trái)`; nếu nhiều `i` cùng đạt max, trả `i` **nhỏ nhất**.

### Bài 4 (tự viết lại)
Viết lại cách 4 nhưng dùng **prefix sum** thay vì cập nhật trượt: `max_score_min_window_pre(cards, k)`.

### Bài 5 (mini challenge)
`max_score_at_most(cards, k)`: điểm có thể **âm**, được lấy **tối đa** `k` lá (kể cả 0 lá). Trả về điểm lớn nhất.

In [28]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def best_split(cards, k):
    n = len(cards)
    pre = list(accumulate(cards, initial=0))
    best, best_i = -1, -1
    for i in range(k + 1):
        # TODO: tính score(i) và cập nhật (best, best_i) khi lớn hơn hẳn
        pass
    return best, best_i

# Bài 4 — TODO
def max_score_min_window_pre(cards, k):
    # TODO: pre; min(pre[i + w] - pre[i]) với i = 0..k; trả total - min
    pass

# Bài 5 — TODO
def max_score_at_most(cards, k):
    # TODO: với mỗi t = 0..k, tính best(t) bằng prefix sum, lấy max
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Liệt kê `i = 0..3`. (2) `i = 0`: `1 + 200 + 1 = 202`; `i = 1`: `1 + 200 + 1 = 202`; `i = 2`: `80 + 1 = 81`; `i = 3`: `160`. (3) Đáp án `202`.

**Bài 2.** (1) `n = 6`, `pre = [0,100,140,157,166,239,314]`. (2) `score = 157, 248, 215, 157`. (3) Đáp án `248` (lấy `100` trái và `73, 75` phải).

**Bài 3.** (1) `s = pre[i] + pre[n] - pre[n - k + i]`. (2) Cập nhật khi `s > best` (dấu `>` giữ `i` nhỏ nhất). (3) Ví dụ 1 trả `(12, 0)`.

**Bài 4.** (1) `w = n - k`. (2) Các cửa sổ `[i, i + w)` với `i ∈ [0, k]`. (3) `w = 0` cũng chạy được vì `pre[i] - pre[i] = 0`.

**Bài 5.** (1) `best(t)` = max với `i ∈ [0, t]` của `pre[i] + pre[n] − pre[n − t + i]`. (2) Kết quả `max(0, best(1), …, best(k))`. (3) `O(k²)` là chấp nhận được cho mini challenge; kiểm chứng bằng vét cạn tập con hợp lệ.

## 19. Đáp án tham khảo

In [29]:
# Bài 1
assert Solution().maxScore([1, 79, 80, 1, 1, 1, 200, 1], 3) == 202
# Bài 2
assert Solution().maxScore([100, 40, 17, 9, 73, 75], 3) == 248

# Bài 3
def best_split_sol(cards, k):
    n = len(cards)
    pre = list(accumulate(cards, initial=0))
    best, best_i = -1, -1
    for i in range(k + 1):
        s = pre[i] + pre[n] - pre[n - k + i]
        if s > best:
            best, best_i = s, i
    return best, best_i

assert best_split_sol([1, 2, 3, 4, 5, 6, 1], 3) == (12, 0)
assert best_split_sol([100, 40, 17, 9, 73, 75], 3) == (248, 1)
assert best_split_sol([2, 2, 2], 2) == (4, 0)

# Bài 4
def max_score_min_window_pre_sol(cards, k):
    n = len(cards)
    pre = list(accumulate(cards, initial=0))
    w = n - k
    return pre[n] - min(pre[i + w] - pre[i] for i in range(k + 1))

# Bài 5
def max_score_at_most_sol(cards, k):
    n = len(cards)
    pre = list(accumulate(cards, initial=0))
    best = 0
    for t in range(1, k + 1):
        for i in range(t + 1):
            best = max(best, pre[i] + pre[n] - pre[n - t + i])
    return best

def brute_at_most(cards, k):
    n = len(cards)
    best = 0
    for t in range(1, k + 1):
        for i in range(t + 1):
            best = max(best, sum(cards[:i]) + sum(cards[n - (t - i):]))
    return best

assert max_score_at_most_sol([-1, -2, -3], 2) == 0
assert max_score_at_most_sol([5, -10, 1, -10, 6], 3) == 11

rng = random.Random(1)
for _ in range(300):
    n = rng.randint(1, 12)
    cards = [rng.randint(1, 30) for _ in range(n)]
    k = rng.randint(1, n)
    assert max_score_min_window_pre_sol(cards, k) == Solution().maxScore(cards, k)
    assert best_split_sol(cards, k)[0] == Solution().maxScore(cards, k)
    neg = [rng.randint(-20, 20) for _ in range(n)]
    assert max_score_at_most_sol(neg, k) == brute_at_most(neg, k)
print("Tất cả đáp án bài tập: OK")

Tất cả đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "lấy hai đầu = bỏ đoạn giữa":**

```python
n, w = len(a), len(a) - k        # w = độ dài đoạn bỏ lại
total = sum(a)
if w == 0: return total
s = mn = sum(a[:w])
for r in range(w, n):
    s += a[r] - a[r - w]
    mn = min(mn, s)
return total - mn
```

**Template trượt trái→phải:**

```python
cur = best = sum(a[:k])
for j in range(1, k + 1):
    cur += a[n - j] - a[k - j]
    best = max(best, cur)
```

**Checklist:**
- [ ] Đã xét đủ `k + 1` phương án (`range(k + 1)`).
- [ ] Không dùng `a[-0:]`.
- [ ] Xử lý `k == n` (cửa sổ rỗng).
- [ ] Cửa sổ bỏ lại dài `n − k`, tìm **min** (không phải max, không phải dài `k`).
- [ ] Không dùng tham lam một bước.
- [ ] Đã thử `[5,1,1,100,1], k=2 → 101` và `[9,9,1,1,1], k=2 → 18`.